# 260908 Charge Fitting Discharge Validation

기준 코드: `260903_0~4단계_HPPC_fitting_C_rate_validation.ipynb`

이번 notebook은 기준 코드의 **Stage 0, Experimental OCP, 1A endpoint-constrained stoichiometry, 공통 DFN builder, solver/mesh/parameter bounds**를 유지하고, 기존 HPPC sensitivity/fitting 부분만 아래 흐름으로 교체한다.

**Stage 0 — Ai2020 source correction**

**Stage 1 — Experimental OCP + 1A endpoint-constrained stoichiometry**
- 음극: 기존 low-rate experimental OCP의 대표 `UN_EXP`
- 양극: 기존 GITT rest-end quasi-OCP의 대표 `UP_EXP`
- Charge/Discharge DFN 모두 동일한 `UN_EXP`, `UP_EXP` 사용
- measured capacity로 `Δx`, `Δy`를 고정
- qOCV 0%/100% endpoint로 `x0`, `y100` 결정

**Stage 2A — Charge-only 7-parameter sensitivity / identifiability**
- `kn, kp, Dsn, Dsp, brugg_n, brugg_p, brugg_s`
- 0.5C / 1C / 2C Charge만 사용

**Stage 2B — Bruggeman = 1.5 고정 후 Charge-only 재분석**
- `kn, kp, Dsn, Dsp`
- sensitivity / identifiability 기반 fitting subset 선정

**Stage 3 — Charge-only fitting + 2개 최적화 전략 비교**
- Algorithm A: Multi-start TRF
- Algorithm B: Differential Evolution → TRF refinement
- 두 알고리즘 모두 동일한 Charge data / parameter subset / bounds / objective 사용
- 알고리즘 비교와 최종 선택에는 Charge fitting 성능만 사용

**Stage 4 — Discharge held-out dynamic validation**
- 각 알고리즘에서 얻은 최종 parameter를 고정한 뒤 0.5C / 1C / 2C Discharge 평가
- Charge = Fitting, Discharge = Validation
- Discharge 성능은 알고리즘의 fitting parameter 선택에 사용하지 않음
- terminal voltage + charge-side anode potential 저장

> `SMOKE_TEST=True`는 실행 경로 확인용이다. 실제 fitting 결과를 얻을 때만 `False`로 변경한다.


In [ ]:
from pathlib import Path
from datetime import datetime
import platform
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import least_squares, differential_evolution

import pybamm

warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style",
)

# ------------------------------------------------------------
# Local VS Code configuration
# ------------------------------------------------------------

ROOT = Path.cwd().resolve()
DATA_DIR_LOCAL = ROOT / "data"
RESULTS_DIR = ROOT / "results" / "charge_fit_discharge_val"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BUNDLE = (
    DATA_DIR_LOCAL
    / "260902 Enertech셀 데이터 모음.xlsx"
)

SMOKE_TEST = True

# Stage 3 compares two optimization strategies:
# A) Multi-start TRF
# B) Differential Evolution -> TRF refinement
OPTIMIZATION_ALGORITHMS = [
    "TRF",
    "DE_TRF",
]

if SMOKE_TEST:
    ACTIVE_RATES = [0.5]
    FIT_SOC_GRID = np.linspace(0.10, 0.90, 31)

    TRF_STARTS = 1
    TRF_MAX_NFEV = 2

    DE_SEEDS = [11]
    DE_MAXITER = 1
    DE_POPSIZE = 3
    DE_REFINE_MAX_NFEV = 2

else:
    ACTIVE_RATES = [0.5, 1.0, 2.0]
    FIT_SOC_GRID = np.linspace(0.10, 0.90, 101)

    TRF_STARTS = 2
    TRF_MAX_NFEV = 25

    # DE는 DFN objective가 비싸므로 첫 비교에서는 1 seed로 제한.
    # 필요하면 결과 검토 후 seed / iteration을 늘린다.
    DE_SEEDS = [11]
    DE_MAXITER = 5
    DE_POPSIZE = 3
    DE_REFINE_MAX_NFEV = 20

MAX_WALL_MIN_PER_TRF_START = 90
MAX_WALL_MIN_PER_DE_SEED = 120
MAX_TOTAL_STAGE3_WALL_MIN = 330

RNG = np.random.default_rng(260908)

plt.rcParams["figure.figsize"] = (7.5, 4.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("=== Environment ===")
print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("PyBaMM :", pybamm.__version__)
print("ROOT   :", ROOT)
print("BUNDLE :", BUNDLE)
print("RESULTS:", RESULTS_DIR)
print("SMOKE_TEST:", SMOKE_TEST)
print("ACTIVE_RATES:", ACTIVE_RATES)
print("OPTIMIZATION_ALGORITHMS:", OPTIMIZATION_ALGORITHMS)

if not BUNDLE.exists():
    raise FileNotFoundError(
        "Input workbook not found. "
        "Repository root에 data/260902 Enertech셀 데이터 모음.xlsx를 두세요."
    )


## 입력 파일

Repository root를 VS Code에서 `Open Folder`로 연 뒤 실행한다.

필수 파일:

`data/260902 Enertech셀 데이터 모음.xlsx`

이 notebook에서는 별도의 HPPC raw Excel을 사용하지 않는다.  
통합 workbook의 `DYNAMIC` 시트에 포함된 0.5C / 1C / 2C Charge·Discharge 데이터를 사용한다.


In [ ]:
print("Input data:", BUNDLE)

# Excel 내부 시트 확인
SHEETS = pd.ExcelFile(BUNDLE).sheet_names
print("Sheets:", SHEETS)

# 각 데이터 시트는 1~7행 source information, 8행 blank, 9행 header 구조
ANODE_LOWRATE_DF = pd.read_excel(
    BUNDLE, sheet_name="ANODE_LOWRATE", header=8
)

ANODE_GITT_DF = pd.read_excel(
    BUNDLE, sheet_name="ANODE_GITT", header=8
)

CATHODE_GITT_DF = pd.read_excel(
    BUNDLE, sheet_name="CATHODE_GITT", header=8
)

FULL_C20_DF = pd.read_excel(
    BUNDLE, sheet_name="FULLCELL_LOWRATE", header=8
)

DYNAMIC_DF = pd.read_excel(
    BUNDLE, sheet_name="DYNAMIC", header=8
)

# PARAMETER_PRIOR는 8행이 header
PRIOR_DF = pd.read_excel(
    BUNDLE, sheet_name="PARAMETER_PRIOR", header=7
)

#key=찾고 싶은 파라미터, 없을경우 None으로 처리, cast=가져온 값을 어떤 자료형으로 바꿀지
def prior(key, default=None, cast=float):
    row = PRIOR_DF[
        PRIOR_DF["parameter"].astype(str) == str(key)
    ]

    if row.empty:
        return default

    value = row.iloc[0]["value"]

    if cast is None:
        return value

    try:
        return cast(value)
    except Exception:
        return default


NOMINAL_CAPACITY_AH = prior("nominal_capacity_Ah")
Q_MEAS_BUNDLE = prior("q_meas_Ah")
T_REF = prior("temperature_K")
DISK_DIAMETER_M = prior("disk_diameter_m")

print("nominal capacity =", NOMINAL_CAPACITY_AH, "Ah")
print("bundle Q_MEAS    =", Q_MEAS_BUNDLE, "Ah")
print("T                =", T_REF, "K")
print("disk diameter    =", DISK_DIAMETER_M, "m")

print("\nLoaded rows")
print("ANODE_LOWRATE   :", len(ANODE_LOWRATE_DF))
print("ANODE_GITT      :", len(ANODE_GITT_DF))
print("CATHODE_GITT    :", len(CATHODE_GITT_DF))
print("FULLCELL        :", len(FULL_C20_DF))
print("DYNAMIC         :", len(DYNAMIC_DF))

## Stage 0. Ai2020 baseline source correction

In [ ]:
DE_MULTIPLIER = 1e-4
CSN_MAX_CORRECTED = 29700.0

# Python 관례
# *args   : 위치 기반(positional) 입력값들을 여러 개 받을 때 사용
# **kwargs: 이름이 지정된(keyword) 입력값들을 여러 개 받을 때 사용
def scale_parameter_function(func, factor):
    def wrapped(*args, **kwargs):
        return factor * func(*args, **kwargs)
    return wrapped


def make_source_corrected_ai2020():
    base = pybamm.ParameterValues("Ai2020")
    pv = base.copy()

    pv.update({
        "Electrolyte diffusivity [m2.s-1]":
            scale_parameter_function(
                base["Electrolyte diffusivity [m2.s-1]"],
                DE_MULTIPLIER,
            ),

        "Maximum concentration in negative electrode [mol.m-3]":
            CSN_MAX_CORRECTED,
    })

    return pv

AI2020_CORRECTED = make_source_corrected_ai2020()

print("=== Stage 0 correction ===")
print("De multiplier =",  DE_MULTIPLIER)
print(
    "c_s,n,max =",
    AI2020_CORRECTED[
        "Maximum concentration in negative electrode [mol.m-3]"
    ],
    "mol m^-3",
)

## Stage 1. Experimental OCP + stoichiometry window

In [ ]:
def extract_qv_branch(df, source_col, source, direction):

    mask = (
        (df[source_col] == source)
        & (df["direction"] == direction)
    )

    selected = df[mask]

    capacity = selected["Q_Ah"].to_numpy(dtype=float)
    voltage = selected["V_V"].to_numpy(dtype=float)

    return capacity, voltage


def extract_gitt_branch(df, source_col, source, pulse_direction):

    selected = df[
        (df[source_col] == source)
        & (df["pulse_direction"] == pulse_direction)
    ]

    capacity = selected["Qcum_Ah"].to_numpy(dtype=float)
    rest_voltage = selected["Vrest_V"].to_numpy(dtype=float)

    return np.c_[capacity, rest_voltage]


# Anode low-rate
# 원본: 260816 Anode GITT + OCP v1.xlsx / 260816 Anode GITT + OCP v2.xlsx
ANODE_V1_CHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v1", "charge"
)

ANODE_V1_DCHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v1", "discharge"
)

ANODE_V2_CHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v2", "charge"
)

ANODE_V2_DCHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v2", "discharge"
)


# Anode GITT
# 원본: 260816 Anode GITT + OCP v1.xlsx / 260816 Anode GITT + OCP v2.xlsx
ANODE_V1_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v1", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v1", "discharge_pulse"
    ),
}

ANODE_V2_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v2", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v2", "discharge_pulse"
    ),
}


# Cathode GITT
# 원본: 260816 Positive GITT 7-7.xlsx / 260816 Positive GITT 7-8.xlsx
CATHODE_77_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-7", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-7", "discharge_pulse"
    ),
}

CATHODE_78_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-8", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-8", "discharge_pulse"
    ),
}


# Full-cell C/20
# 원본: 260808 Enertech셀 초기 저율 데이터 v2.xlsx
FULLCELL_V2_CHG_BRANCH = extract_qv_branch(
    FULL_C20_DF, "source_version", "v2", "charge"
)

FULLCELL_V2_DCHG_BRANCH = extract_qv_branch(
    FULL_C20_DF, "source_version", "v2", "discharge"
)


print("Standardized Stage 1 data loaded.")

print(
    "Anode low-rate v1:",
    len(ANODE_V1_CHG_BRANCH[0]),
    len(ANODE_V1_DCHG_BRANCH[0])
)

print(
    "Anode low-rate v2:",
    len(ANODE_V2_CHG_BRANCH[0]),
    len(ANODE_V2_DCHG_BRANCH[0])
)

print(
    "Anode GITT v1:",
    len(ANODE_V1_GITT["CHG_BRANCH"]),
    len(ANODE_V1_GITT["DCHG_BRANCH"])
)

print(
    "Anode GITT v2:",
    len(ANODE_V2_GITT["CHG_BRANCH"]),
    len(ANODE_V2_GITT["DCHG_BRANCH"])
)

print(
    "Cathode GITT 7-7:",
    len(CATHODE_77_GITT["CHG_BRANCH"]),
    len(CATHODE_77_GITT["DCHG_BRANCH"])
)

print(
    "Cathode GITT 7-8:",
    len(CATHODE_78_GITT["CHG_BRANCH"]),
    len(CATHODE_78_GITT["DCHG_BRANCH"])
)

print(
    "Full-cell v2 C/20:",
    len(FULLCELL_V2_CHG_BRANCH[0]),
    len(FULLCELL_V2_DCHG_BRANCH[0])
)

In [ ]:
# ============================================================
# Cell 1. Capacity → Stoichiometry 변환
# ============================================================

FARADAY_CONSTANT = 96485.33212


# Maximum solid concentration
CSN_MAX = float(
    AI2020_CORRECTED[
        "Maximum concentration in negative electrode [mol.m-3]"
    ]
)

CSP_MAX = float(
    AI2020_CORRECTED[
        "Maximum concentration in positive electrode [mol.m-3]"
    ]
)


# Active material volume fraction
EPS_N = float(
    AI2020_CORRECTED[
        "Negative electrode active material volume fraction"
    ]
)

EPS_P = float(
    AI2020_CORRECTED[
        "Positive electrode active material volume fraction"
    ]
)


# Electrode thickness
LN = float(
    AI2020_CORRECTED[
        "Negative electrode thickness [m]"
    ]
)

LP = float(
    AI2020_CORRECTED[
        "Positive electrode thickness [m]"
    ]
)


# Ø10 mm half-cell electrode disk area
DISK_AREA = np.pi * (DISK_DIAMETER_M / 2.0)**2


# Half-cell electrode 기준 용량 [Ah]
QN_DISK_AH = (
    FARADAY_CONSTANT
    * CSN_MAX
    * EPS_N
    * LN
    * DISK_AREA
    / 3600
)

QP_DISK_AH = (
    FARADAY_CONSTANT
    * CSP_MAX
    * EPS_P
    * LP
    * DISK_AREA
    / 3600
)


def anode_stoich_axis(branch, direction):
    capacity, voltage = branch

    if direction == "lith":
        x = capacity / QN_DISK_AH

    else:
        span = capacity.max() / QN_DISK_AH
        x = span - capacity / QN_DISK_AH

    order = np.argsort(x)

    return x[order], voltage[order]


def cathode_stoich_axis(branch, direction):
    capacity = branch[:, 0]
    voltage = branch[:, 1]

    if direction == "lith":
        span = capacity.max() / QP_DISK_AH
        y = (1 - span) + capacity / QP_DISK_AH

    else:
        y = 1 - capacity / QP_DISK_AH

    order = np.argsort(y)

    return y[order], voltage[order]


# ------------------------------------------------------------
# Anode stoichiometry 변환
# Graphite/Li half-cell
# DCHG → lithiation   (Li insertion, x 증가)
# CHG  → delithiation (Li extraction, x 감소)
#
# 원본:
# 260816 Anode GITT + OCP v1.xlsx
# 260816 Anode GITT + OCP v2.xlsx
# ------------------------------------------------------------

ANODE_V1_LITH = anode_stoich_axis(
    ANODE_V1_DCHG_BRANCH,
    "lith"
)

ANODE_V1_DELITH = anode_stoich_axis(
    ANODE_V1_CHG_BRANCH,
    "delith"
)

ANODE_V2_LITH = anode_stoich_axis(
    ANODE_V2_DCHG_BRANCH,
    "lith"
)

ANODE_V2_DELITH = anode_stoich_axis(
    ANODE_V2_CHG_BRANCH,
    "delith"
)


# ------------------------------------------------------------
# Cathode stoichiometry 변환
# LCO/Li half-cell
# DCHG pulse → lithiation   (Li insertion, y 증가)
# CHG pulse  → delithiation (Li extraction, y 감소)
#
# 원본:
# 260816 Positive GITT 7-7.xlsx
# 260816 Positive GITT 7-8.xlsx
# ------------------------------------------------------------

CATHODE_77_LITH = cathode_stoich_axis(
    CATHODE_77_GITT["DCHG_BRANCH"],
    "lith"
)

CATHODE_77_DELITH = cathode_stoich_axis(
    CATHODE_77_GITT["CHG_BRANCH"],
    "delith"
)

CATHODE_78_LITH = cathode_stoich_axis(
    CATHODE_78_GITT["DCHG_BRANCH"],
    "lith"
)

CATHODE_78_DELITH = cathode_stoich_axis(
    CATHODE_78_GITT["CHG_BRANCH"],
    "delith"
)


print("QN disk capacity =", QN_DISK_AH, "Ah")
print("QP disk capacity =", QP_DISK_AH, "Ah")

In [ ]:
# ============================================================
# Cell 2. OCP / Stoichiometry 변환 결과 확인
# ============================================================


# ------------------------------------------------------------
# 1. Anode 원본 Capacity-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    ANODE_V1_CHG_BRANCH[0],
    ANODE_V1_CHG_BRANCH[1],
    label="v1 CHG"
)

plt.plot(
    ANODE_V1_DCHG_BRANCH[0],
    ANODE_V1_DCHG_BRANCH[1],
    label="v1 DCHG"
)

plt.plot(
    ANODE_V2_CHG_BRANCH[0],
    ANODE_V2_CHG_BRANCH[1],
    label="v2 CHG"
)

plt.plot(
    ANODE_V2_DCHG_BRANCH[0],
    ANODE_V2_DCHG_BRANCH[1],
    label="v2 DCHG"
)

plt.xlabel("Capacity [Ah]")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title("Anode half-cell: Original Q-V curves")

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.02,
    "Source: 260816 Anode GITT + OCP v1.xlsx / "
    "260816 Anode GITT + OCP v2.xlsx",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Anode Stoichiometry-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    ANODE_V1_LITH[0],
    ANODE_V1_LITH[1],
    label="v1 lith (DCHG)"
)

plt.plot(
    ANODE_V1_DELITH[0],
    ANODE_V1_DELITH[1],
    label="v1 delith (CHG)"
)

plt.plot(
    ANODE_V2_LITH[0],
    ANODE_V2_LITH[1],
    label="v2 lith (DCHG)"
)

plt.plot(
    ANODE_V2_DELITH[0],
    ANODE_V2_DELITH[1],
    label="v2 delith (CHG)"
)

plt.xlabel("Anode stoichiometry x")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title(
    "Anode half-cell: Stoichiometry-converted OCP curves"
)

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.04,
    "Source: 260816 Anode GITT + OCP v1.xlsx / "
    "260816 Anode GITT + OCP v2.xlsx\n"
    "Graphite/Li: DCHG → lithiation, CHG → delithiation",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. Cathode 원본 GITT Capacity-Rest Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    CATHODE_77_GITT["CHG_BRANCH"][:, 0],
    CATHODE_77_GITT["CHG_BRANCH"][:, 1],
    marker="o",
    label="7-7 CHG pulse"
)

plt.plot(
    CATHODE_77_GITT["DCHG_BRANCH"][:, 0],
    CATHODE_77_GITT["DCHG_BRANCH"][:, 1],
    marker="o",
    label="7-7 DCHG pulse"
)

plt.plot(
    CATHODE_78_GITT["CHG_BRANCH"][:, 0],
    CATHODE_78_GITT["CHG_BRANCH"][:, 1],
    marker="o",
    label="7-8 CHG pulse"
)

plt.plot(
    CATHODE_78_GITT["DCHG_BRANCH"][:, 0],
    CATHODE_78_GITT["DCHG_BRANCH"][:, 1],
    marker="o",
    label="7-8 DCHG pulse"
)

plt.xlabel("Cumulative capacity [Ah]")
plt.ylabel("Rest voltage [V vs. Li/Li+]")
plt.title("Cathode half-cell: Original GITT OCP points")

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.02,
    "Source: 260816 Positive GITT 7-7.xlsx / "
    "260816 Positive GITT 7-8.xlsx",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 4. Cathode Stoichiometry-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    CATHODE_77_LITH[0],
    CATHODE_77_LITH[1],
    marker="o",
    label="7-7 lith (DCHG)"
)

plt.plot(
    CATHODE_77_DELITH[0],
    CATHODE_77_DELITH[1],
    marker="o",
    label="7-7 delith (CHG)"
)

plt.plot(
    CATHODE_78_LITH[0],
    CATHODE_78_LITH[1],
    marker="o",
    label="7-8 lith (DCHG)"
)

plt.plot(
    CATHODE_78_DELITH[0],
    CATHODE_78_DELITH[1],
    marker="o",
    label="7-8 delith (CHG)"
)

plt.xlabel("Cathode stoichiometry y")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title(
    "Cathode half-cell: Stoichiometry-converted OCP curves"
)

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.04,
    "Source: 260816 Positive GITT 7-7.xlsx / "
    "260816 Positive GITT 7-8.xlsx\n"
    "LCO/Li: DCHG → lithiation, CHG → delithiation",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Stoichiometry range 확인
# ------------------------------------------------------------

print("=== Anode stoichiometry range ===")

print(
    "v1 lith   :",
    ANODE_V1_LITH[0].min(),
    "to",
    ANODE_V1_LITH[0].max()
)

print(
    "v1 delith :",
    ANODE_V1_DELITH[0].min(),
    "to",
    ANODE_V1_DELITH[0].max()
)

print(
    "v2 lith   :",
    ANODE_V2_LITH[0].min(),
    "to",
    ANODE_V2_LITH[0].max()
)

print(
    "v2 delith :",
    ANODE_V2_DELITH[0].min(),
    "to",
    ANODE_V2_DELITH[0].max()
)


print("\n=== Cathode stoichiometry range ===")

print(
    "7-7 lith   :",
    CATHODE_77_LITH[0].min(),
    "to",
    CATHODE_77_LITH[0].max()
)

print(
    "7-7 delith :",
    CATHODE_77_DELITH[0].min(),
    "to",
    CATHODE_77_DELITH[0].max()
)

print(
    "7-8 lith   :",
    CATHODE_78_LITH[0].min(),
    "to",
    CATHODE_78_LITH[0].max()
)

print(
    "7-8 delith :",
    CATHODE_78_DELITH[0].min(),
    "to",
    CATHODE_78_DELITH[0].max()
)


# ------------------------------------------------------------
# 6. 여러 OCP curve의 공통 범위에서 평균 curve 생성
# ------------------------------------------------------------

def mean_overlap(curves, n=3000):

    lower_limit = max(
        x.min() for x, _ in curves
    )

    upper_limit = min(
        x.max() for x, _ in curves
    )

    assert upper_limit > lower_limit, (
        "OCP curve common overlap이 없습니다."
    )

    grid = np.linspace(
        lower_limit,
        upper_limit,
        n
    )

    values = np.vstack([
        np.interp(grid, x, y)
        for x, y in curves
    ])

    return grid, values.mean(axis=0)

In [ ]:
# ============================================================
# Experimental OCP 구성 + Ai2020 비교
# ============================================================


# ------------------------------------------------------------
# 1. Experimental Anode OCP
#
# 현재 backbone:
# v1 / v2 low-rate OCP
# ------------------------------------------------------------

X_LITH, UN_LITH = mean_overlap([
    ANODE_V1_LITH,
    ANODE_V2_LITH,
])

X_DELITH, UN_DELITH = mean_overlap([
    ANODE_V1_DELITH,
    ANODE_V2_DELITH,
])


X_MIN = max(
    X_LITH.min(),
    X_DELITH.min()
)

X_MAX = min(
    X_LITH.max(),
    X_DELITH.max()
)

X_EXP = np.linspace(
    X_MIN,
    X_MAX,
    3000
)


UN_EXP = 0.5 * (
    np.interp(
        X_EXP,
        X_LITH,
        UN_LITH
    )
    +
    np.interp(
        X_EXP,
        X_DELITH,
        UN_DELITH
    )
)


# ------------------------------------------------------------
# 2. Experimental Cathode OCP
#
# 현재 backbone:
# GITT 7-7 / 7-8 rest-end quasi-OCP
# ------------------------------------------------------------

Y_EXP, UP_EXP = mean_overlap([
    CATHODE_77_LITH,
    CATHODE_78_LITH,
    CATHODE_77_DELITH,
    CATHODE_78_DELITH,
])


# ------------------------------------------------------------
# 3. Ai2020 OCP 불러오기
# ------------------------------------------------------------

PYBAMM_DIR = Path(
    pybamm.__file__
).resolve().parent

DATA_DIR = (
    PYBAMM_DIR
    / "input"
    / "parameters"
    / "lithium_ion"
    / "data"
)


def read_ai2020_ocp(filename):

    data = np.genfromtxt(
        DATA_DIR / filename,
        delimiter=","
    )

    data = data[
        np.isfinite(data).all(axis=1)
    ]

    return data[:, 0], data[:, 1]


X_AI, UN_AI = read_ai2020_ocp(
    "graphite_ocp_Enertech_Ai2020.csv"
)

Y_AI, UP_AI = read_ai2020_ocp(
    "lico2_ocp_Ai2020.csv"
)


# ------------------------------------------------------------
# 4. Numerical OCP 함수
#
# 뒤의 stoichiometry window fitting에서 사용
# ------------------------------------------------------------

UN_AI_SPLINE = CubicSpline(
    X_AI,
    UN_AI,
    extrapolate=True
)

UP_AI_SPLINE = CubicSpline(
    Y_AI,
    UP_AI,
    extrapolate=True
)


def Un_ai(x):
    return UN_AI_SPLINE(x)


def Up_ai(y):
    return UP_AI_SPLINE(y)


def Un_exp(x):
    return np.interp(
        x,
        X_EXP,
        UN_EXP
    )


def Up_exp(y):
    return np.interp(
        y,
        Y_EXP,
        UP_EXP
    )


# ------------------------------------------------------------
# 5. OCP 자체 비교
# ------------------------------------------------------------

fig, ax = plt.subplots(
    1,
    2,
    figsize=(11, 4.4)
)


ax[0].plot(
    X_AI,
    UN_AI,
    lw=2,
    label="Ai2020"
)

ax[0].plot(
    X_EXP,
    UN_EXP,
    lw=2,
    label="Experimental"
)

ax[0].set_title("Anode OCP")
ax[0].set_xlabel("Graphite stoichiometry x")
ax[0].set_ylabel("Voltage [V vs. Li/Li+]")
ax[0].legend()


ax[1].plot(
    Y_AI,
    UP_AI,
    lw=2,
    label="Ai2020"
)

ax[1].plot(
    Y_EXP,
    UP_EXP,
    lw=2,
    label="Experimental"
)

ax[1].set_title("Cathode OCP")
ax[1].set_xlabel("LCO stoichiometry y")
ax[1].set_ylabel("Voltage [V vs. Li/Li+]")
ax[1].legend()


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Experimental OCP → PyBaMM parameter에 실제 주입
# ============================================================


def pybamm_interp(grid, values, name):

    grid = np.asarray(
        grid,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )

    def ocp(sto):

        return pybamm.Interpolant(
            grid,
            values,
            [sto],
            name=name,
            interpolator="linear",
            extrapolate=True,
        )

    return ocp


# Stage 0 correction이 적용된 Ai2020 복사
OCP_REPLACED_PARAMS = AI2020_CORRECTED.copy()


# OCP 두 개만 Experimental로 교체
OCP_REPLACED_PARAMS.update({

    "Negative electrode OCP [V]":
        pybamm_interp(
            X_EXP,
            UN_EXP,
            "Experimental anode OCP",
        ),

    "Positive electrode OCP [V]":
        pybamm_interp(
            Y_EXP,
            UP_EXP,
            "Experimental cathode OCP",
        ),
})


print("=== OCP replacement ===")

print(
    "Anode OCP replaced:",
    OCP_REPLACED_PARAMS[
        "Negative electrode OCP [V]"
    ]
    is not
    AI2020_CORRECTED[
        "Negative electrode OCP [V]"
    ]
)

print(
    "Cathode OCP replaced:",
    OCP_REPLACED_PARAMS[
        "Positive electrode OCP [V]"
    ]
    is not
    AI2020_CORRECTED[
        "Positive electrode OCP [V]"
    ]
)

print(
    "Experimental anode coverage:",
    X_EXP.min(),
    "to",
    X_EXP.max()
)

print(
    "Experimental cathode coverage:",
    Y_EXP.min(),
    "to",
    Y_EXP.max()
)

In [ ]:
# ============================================================
# OCP 교체 전 / 후 차이 확인
# ============================================================


# Experimental data가 존재하는 공통 영역에서만 비교
X_COMPARE = np.linspace(
    max(X_AI.min(), X_EXP.min()),
    min(X_AI.max(), X_EXP.max()),
    500
)

Y_COMPARE = np.linspace(
    max(Y_AI.min(), Y_EXP.min()),
    min(Y_AI.max(), Y_EXP.max()),
    500
)


UN_BEFORE = Un_ai(X_COMPARE)
UN_AFTER = Un_exp(X_COMPARE)

UP_BEFORE = Up_ai(Y_COMPARE)
UP_AFTER = Up_exp(Y_COMPARE)


DELTA_UN_MV = (
    UN_AFTER - UN_BEFORE
) * 1000

DELTA_UP_MV = (
    UP_AFTER - UP_BEFORE
) * 1000


fig, ax = plt.subplots(
    2,
    2,
    figsize=(11, 7)
)


# Anode OCP
ax[0, 0].plot(
    X_COMPARE,
    UN_BEFORE,
    label="Before: Ai2020"
)

ax[0, 0].plot(
    X_COMPARE,
    UN_AFTER,
    label="After: Experimental"
)

ax[0, 0].set_title("Anode OCP replacement")
ax[0, 0].set_xlabel("Graphite stoichiometry x")
ax[0, 0].set_ylabel("Voltage [V]")
ax[0, 0].legend()


# Cathode OCP
ax[0, 1].plot(
    Y_COMPARE,
    UP_BEFORE,
    label="Before: Ai2020"
)

ax[0, 1].plot(
    Y_COMPARE,
    UP_AFTER,
    label="After: Experimental"
)

ax[0, 1].set_title("Cathode OCP replacement")
ax[0, 1].set_xlabel("LCO stoichiometry y")
ax[0, 1].set_ylabel("Voltage [V]")
ax[0, 1].legend()


# Anode difference
ax[1, 0].plot(
    X_COMPARE,
    DELTA_UN_MV
)

ax[1, 0].axhline(
    0,
    linewidth=1
)

ax[1, 0].set_title(
    "Anode ΔOCP"
)

ax[1, 0].set_xlabel(
    "Graphite stoichiometry x"
)

ax[1, 0].set_ylabel(
    "Experimental - Ai2020 [mV]"
)


# Cathode difference
ax[1, 1].plot(
    Y_COMPARE,
    DELTA_UP_MV
)

ax[1, 1].axhline(
    0,
    linewidth=1
)

ax[1, 1].set_title(
    "Cathode ΔOCP"
)

ax[1, 1].set_xlabel(
    "LCO stoichiometry y"
)

ax[1, 1].set_ylabel(
    "Experimental - Ai2020 [mV]"
)


plt.tight_layout()
plt.show()


print(
    "Anode mean |ΔOCP| =",
    np.mean(np.abs(DELTA_UN_MV)),
    "mV"
)

print(
    "Anode max |ΔOCP|  =",
    np.max(np.abs(DELTA_UN_MV)),
    "mV"
)

print(
    "Cathode mean |ΔOCP| =",
    np.mean(np.abs(DELTA_UP_MV)),
    "mV"
)

print(
    "Cathode max |ΔOCP|  =",
    np.max(np.abs(DELTA_UP_MV)),
    "mV"
)

In [ ]:
# ============================================================
# Full-cell C/20 qOCV + electrode capacity
# ============================================================

SOC = np.linspace(0, 1, 1001)


def interp_unique(x, y, xnew):

    order = np.argsort(x)

    x = np.asarray(x)[order]
    y = np.asarray(y)[order]

    unique_x, index = np.unique(
        x,
        return_index=True
    )

    return np.interp(
        xnew,
        unique_x,
        y[index]
    )


# ------------------------------------------------------------
# 1. Full-cell C/20 CHG / DCHG
#
# 원본:
# 260808 Enertech셀 초기 저율 데이터 v2.xlsx
# ------------------------------------------------------------

capacity_chg, voltage_chg = FULLCELL_V2_CHG_BRANCH
capacity_dchg, voltage_dchg = FULLCELL_V2_DCHG_BRANCH


# Charge / discharge를 동일한 SOC 축으로 변환
V_CHG = interp_unique(
    capacity_chg / capacity_chg.max(),
    voltage_chg,
    SOC
)

V_DCH = interp_unique(
    1 - capacity_dchg / capacity_dchg.max(),
    voltage_dchg,
    SOC
)


# CHG / DCHG 평균 → full-cell quasi-OCV
V_QOCV = 0.5 * (
    V_CHG + V_DCH
)


# 측정 full-cell capacity
Q_MEAS = 0.5 * (
    capacity_chg.max()
    + capacity_dchg.max()
)


print(
    f"Bundle Q_MEAS={Q_MEAS_BUNDLE:.6f} Ah | "
    f"recomputed={Q_MEAS:.6f} Ah | "
    f"Δ={(Q_MEAS-Q_MEAS_BUNDLE)*1000:.3f} mAh"
)


# ------------------------------------------------------------
# 2. Ai2020 geometry 기반 electrode theoretical capacity
# ------------------------------------------------------------

CELL_AREA = (
    float(
        AI2020_CORRECTED[
            "Electrode height [m]"
        ]
    )
    *
    float(
        AI2020_CORRECTED[
            "Electrode width [m]"
        ]
    )
    *
    float(
        AI2020_CORRECTED[
            "Number of electrodes connected in parallel to make a cell"
        ]
    )
)


QN_CELL = (
    FARADAY_CONSTANT
    * CSN_MAX
    * EPS_N
    * LN
    * CELL_AREA
    / 3600
)


QP_CELL = (
    FARADAY_CONSTANT
    * CSP_MAX
    * EPS_P
    * LP
    * CELL_AREA
    / 3600
)


# ------------------------------------------------------------
# 3. Full-cell measured capacity에 해당하는 stoichiometry span
# ------------------------------------------------------------

DX = Q_MEAS / QN_CELL
DY = Q_MEAS / QP_CELL


print(f"Q_MEAS = {Q_MEAS:.6f} Ah")

print(
    f"Qn theoretical = {QN_CELL:.6f} Ah, "
    f"Δx = {DX:.6f}"
)

print(
    f"Qp theoretical = {QP_CELL:.6f} Ah, "
    f"Δy = {DY:.6f}"
)


assert 0 < DX < 1
assert 0 < DY < 1

# Stage 1 — Experimental OCP + 1A endpoint-constrained stoichiometry

Stage 1 방법론은 별도 `1A–1D validation notebook`에서 검토했고, 메인 모델에서는 **1A Endpoint-constrained balancing**을 사용한다.

고정되는 span:

\[
\Delta x = \frac{Q_{cell}}{Q_n}, \qquad
\Delta y = \frac{Q_{cell}}{Q_p}
\]

\[
x_{100}=x_0+\Delta x,\qquad
y_0=y_{100}+\Delta y
\]

`x0`, `y100`은 전체 qOCV RMSE를 최소화해서 찾지 않고, full-cell qOCV endpoint 조건을 만족하도록 결정한다.

\[
U_p(y_0)-U_n(x_0)=V_0
\]

\[
U_p(y_{100})-U_n(x_{100})=V_{100}
\]

RMSE / MAE는 결정된 window의 **검증 지표**로 계산한다.


In [ ]:
# ============================================================
# Stage 1 — 1A endpoint-constrained stoichiometry
# ============================================================

V0_TARGET = float(V_QOCV[0])
V100_TARGET = float(V_QOCV[-1])


def make_window_voltage(
    x0,
    x100,
    y100,
    y0,
):
    x = x0 + SOC * (x100 - x0)
    y = y0 - SOC * (y0 - y100)

    return (
        Up_exp(y)
        - Un_exp(x)
    )


def window_metrics(
    voltage,
):
    error_mV = (
        voltage
        - V_QOCV
    ) * 1000

    mid_mask = (
        (SOC >= 0.02)
        & (SOC <= 0.98)
    )

    return {
        "RMSE_full_mV":
            np.sqrt(
                np.mean(error_mV**2)
            ),

        "MAE_full_mV":
            np.mean(
                np.abs(error_mV)
            ),

        "RMSE_2_98_mV":
            np.sqrt(
                np.mean(
                    error_mV[mid_mask]**2
                )
            ),

        "MAE_2_98_mV":
            np.mean(
                np.abs(
                    error_mV[mid_mask]
                )
            ),
    }


def solve_stage1_endpoint():

    x0_lower = max(
        1e-6,
        float(X_EXP.min()),
    )

    x0_upper = min(
        1.0 - DX - 1e-6,
        float(X_EXP.max()) - DX,
    )

    y100_lower = max(
        1e-6,
        float(Y_EXP.min()),
    )

    y100_upper = min(
        1.0 - DY - 1e-6,
        float(Y_EXP.max()) - DY,
    )

    if (
        x0_upper <= x0_lower
        or y100_upper <= y100_lower
    ):
        raise RuntimeError(
            "Experimental OCP coverage가 "
            "필요 stoichiometry span을 덮지 못합니다."
        )

    def residuals(offsets):

        x0, y100 = offsets

        x100 = (
            x0 + DX
        )

        y0 = (
            y100 + DY
        )

        voltage_0 = (
            Up_exp(y0)
            - Un_exp(x0)
        )

        voltage_100 = (
            Up_exp(y100)
            - Un_exp(x100)
        )

        return np.array([
            voltage_0 - V0_TARGET,
            voltage_100 - V100_TARGET,
        ])

    initial_guess = np.array([
        0.5 * (
            x0_lower
            + x0_upper
        ),
        0.5 * (
            y100_lower
            + y100_upper
        ),
    ])

    result = least_squares(
        residuals,
        x0=initial_guess,
        bounds=(
            [
                x0_lower,
                y100_lower,
            ],
            [
                x0_upper,
                y100_upper,
            ],
        ),
        xtol=1e-12,
        ftol=1e-12,
        gtol=1e-12,
    )

    x0, y100 = result.x

    x100 = (
        x0 + DX
    )

    y0 = (
        y100 + DY
    )

    voltage = make_window_voltage(
        x0,
        x100,
        y100,
        y0,
    )

    metrics = window_metrics(
        voltage
    )

    endpoint_error_mV = (
        residuals(result.x)
        * 1000
    )

    return {
        "x0":
            float(x0),

        "x100":
            float(x100),

        "y100":
            float(y100),

        "y0":
            float(y0),

        "DX":
            float(DX),

        "DY":
            float(DY),

        "Endpoint_0_error_mV":
            float(
                endpoint_error_mV[0]
            ),

        "Endpoint_100_error_mV":
            float(
                endpoint_error_mV[1]
            ),

        **metrics,

        "Solver_success":
            bool(result.success),
    }


STAGE1 = solve_stage1_endpoint()


VOLTAGE_STAGE1 = make_window_voltage(
    STAGE1["x0"],
    STAGE1["x100"],
    STAGE1["y100"],
    STAGE1["y0"],
)


STAGE1_TABLE = pd.DataFrame([
    {
        "Method":
            "1A Endpoint-constrained",

        "Qn_Ah":
            QN_CELL,

        "Qp_Ah":
            QP_CELL,

        **STAGE1,
    }
])


display(
    STAGE1_TABLE.round(5)
)


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    SOC,
    V_QOCV,
    lw=2.5,
    label="Measured C/20 qOCV",
)

plt.plot(
    SOC,
    VOLTAGE_STAGE1,
    lw=2,
    label="Stage 1 — 1A",
)

plt.xlabel("SOC")
plt.ylabel("Cell OCV [V]")
plt.title("Stage 1 qOCV validation")
plt.legend()
plt.grid()

plt.show()


## 공통 DFN parameter builder

Stage 1에서 확정한 Experimental OCP와 1A stoichiometry는 이후 Stage 2–4에서 고정한다.

추정 후보:
- `kn_mult`
- `kp_mult`
- `Dsn_mult`
- `Dsp_mult`
- `brugg_n`
- `brugg_p`
- `brugg_s`

Stage 2–3에서 OCP / stoichiometry는 다시 fitting하지 않는다.


In [ ]:
# ============================================================
# Common DFN parameter builder
# Stage 1 OCP + stoichiometry fixed
# ============================================================

BRUGG_N_AI2020 = float(
    AI2020_CORRECTED[
        "Negative electrode Bruggeman coefficient (electrolyte)"
    ]
)

BRUGG_P_AI2020 = float(
    AI2020_CORRECTED[
        "Positive electrode Bruggeman coefficient (electrolyte)"
    ]
)

BRUGG_S_AI2020 = float(
    AI2020_CORRECTED[
        "Separator Bruggeman coefficient (electrolyte)"
    ]
)


PARS_NOMINAL = {
    "kn_mult": 1.0,
    "kp_mult": 1.0,
    "Dsn_mult": 1.0,
    "Dsp_mult": 1.0,
    "brugg_n": BRUGG_N_AI2020,
    "brugg_p": BRUGG_P_AI2020,
    "brugg_s": BRUGG_S_AI2020,
}


PARS_BRUGG_15 = (
    PARS_NOMINAL.copy()
)

PARS_BRUGG_15.update({
    "brugg_n": 1.5,
    "brugg_p": 1.5,
    "brugg_s": 1.5,
})


PARAMS_7 = [
    "kn_mult",
    "kp_mult",
    "Dsn_mult",
    "Dsp_mult",
    "brugg_n",
    "brugg_p",
    "brugg_s",
]


PARAMS_4 = [
    "kn_mult",
    "kp_mult",
    "Dsn_mult",
    "Dsp_mult",
]


PARAM_LABELS = {
    "kn_mult": "k_n",
    "kp_mult": "k_p",
    "Dsn_mult": "D_s,n",
    "Dsp_mult": "D_s,p",
    "brugg_n": "b_n",
    "brugg_p": "b_p",
    "brugg_s": "b_s",
}


PARAM_BOUNDS = {
    "kn_mult": (0.20, 5.00),
    "kp_mult": (0.20, 5.00),
    "Dsn_mult": (0.10, 10.00),
    "Dsp_mult": (0.10, 10.00),
    "brugg_n": (1.00, 4.00),
    "brugg_p": (1.00, 4.00),
    "brugg_s": (1.00, 4.00),
}


def make_dfn_parameter_values(
    x_init,
    y_init,
    pars,
):

    base = (
        AI2020_CORRECTED.copy()
    )

    pv = base.copy()


    neg_j0_base = base[
        "Negative electrode exchange-current density [A.m-2]"
    ]

    pos_j0_base = base[
        "Positive electrode exchange-current density [A.m-2]"
    ]

    dsn_base = base[
        "Negative particle diffusivity [m2.s-1]"
    ]

    dsp_base = base[
        "Positive particle diffusivity [m2.s-1]"
    ]


    pv.update({

        "Initial concentration in negative electrode [mol.m-3]":
            float(x_init)
            * CSN_MAX,

        "Initial concentration in positive electrode [mol.m-3]":
            float(y_init)
            * CSP_MAX,


        "Negative electrode OCP [V]":
            pybamm_interp(
                X_EXP,
                UN_EXP,
                "Experimental negative OCP",
            ),

        "Positive electrode OCP [V]":
            pybamm_interp(
                Y_EXP,
                UP_EXP,
                "Experimental positive OCP",
            ),


        "Negative electrode exchange-current density [A.m-2]":
            scale_parameter_function(
                neg_j0_base,
                pars["kn_mult"],
            ),

        "Positive electrode exchange-current density [A.m-2]":
            scale_parameter_function(
                pos_j0_base,
                pars["kp_mult"],
            ),


        "Negative particle diffusivity [m2.s-1]":
            scale_parameter_function(
                dsn_base,
                pars["Dsn_mult"],
            ),

        "Positive particle diffusivity [m2.s-1]":
            scale_parameter_function(
                dsp_base,
                pars["Dsp_mult"],
            ),


        "Negative electrode Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_n"]
            ),

        "Positive electrode Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_p"]
            ),

        "Separator Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_s"]
            ),


        "Initial temperature [K]":
            T_REF,

        "Ambient temperature [K]":
            T_REF,

    }, check_already_exists=False)


    return pv


print(
    "Ai2020 Bruggeman:",
    BRUGG_N_AI2020,
    BRUGG_P_AI2020,
    BRUGG_S_AI2020,
)


In [ ]:
# Stage 1 결과 저장
STAGE1_TABLE.to_csv(
    RESULTS_DIR / "stage1_stoichiometry.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved:",
    RESULTS_DIR / "stage1_stoichiometry.csv",
)


## Stage 1 baseline — Charge dynamic response

이 단계는 parameter fitting이 아니다.

Stage 1에서 확정한 Experimental OCP + 1A stoichiometry만 적용했을 때,  
현재 fitting에 사용할 Charge 조건에서 nominal DFN이 정상적으로 작동하는지 확인한다.

Discharge dynamic profile은 이 단계에서 평가하지 않고 Stage 4 held-out validation에서 처음 평가한다.


In [ ]:
# ============================================================
# Dynamic experimental data + common DFN simulation
# ============================================================

def load_dynamic_from_bundle():

    out = {}

    for rate in [0.5, 1.0, 2.0]:

        for charge in [True, False]:

            direction = (
                "charge"
                if charge
                else "discharge"
            )

            selected = DYNAMIC_DF[
                np.isclose(
                    pd.to_numeric(
                        DYNAMIC_DF["rate_C"],
                        errors="coerce",
                    ),
                    rate,
                )
                &
                (
                    DYNAMIC_DF["direction"]
                    .astype(str)
                    .str.lower()
                    == direction
                )
            ].copy()

            assert (
                len(selected) > 5
            ), (
                f"Dynamic data missing: "
                f"{rate}C {direction}"
            )

            out[(rate, charge)] = {
                "t_min":
                    pd.to_numeric(
                        selected["t_min"],
                        errors="coerce",
                    ).to_numpy(dtype=float),

                "V":
                    pd.to_numeric(
                        selected["V_V"],
                        errors="coerce",
                    ).to_numpy(dtype=float),

                "SOC":
                    pd.to_numeric(
                        selected["SOC"],
                        errors="coerce",
                    ).to_numpy(dtype=float),
            }

    return out


EXP_DYN = load_dynamic_from_bundle()


def run_cc_dfn(
    rate,
    charge,
    pars,
):

    x_init = (
        STAGE1["x0"]
        if charge
        else STAGE1["x100"]
    )

    y_init = (
        STAGE1["y0"]
        if charge
        else STAGE1["y100"]
    )

    model = pybamm.lithium_ion.DFN({
        "thermal": "isothermal",
    })

    model.variables[
        "Anode potential [V]"
    ] = model.variables[
        "Negative electrode surface potential difference at separator interface [V]"
    ]

    step = (
        f"Charge at {rate}C until 4.2 V"
        if charge
        else
        f"Discharge at {rate}C until 3.0 V"
    )

    sim = pybamm.Simulation(
        model,

        parameter_values=
            make_dfn_parameter_values(
                x_init,
                y_init,
                pars,
            ),

        experiment=
            pybamm.Experiment(
                [step],
                period="10 seconds",
            ),

        var_pts={
            "x_n": 15,
            "x_s": 15,
            "x_p": 15,
            "r_n": 15,
            "r_p": 15,
        },

        solver=
            pybamm.IDAKLUSolver(
                rtol=1e-6,
                atol=1e-8,
            ),
    )

    sol = sim.solve()

    time_s = np.asarray(
        sol["Time [s]"].entries,
        dtype=float,
    )

    voltage = np.asarray(
        sol["Terminal voltage [V]"].entries,
        dtype=float,
    )

    current = np.asarray(
        sol["Current [A]"].entries,
        dtype=float,
    )

    anode_potential = np.asarray(
        sol["Anode potential [V]"].entries,
        dtype=float,
    ).squeeze()

    capacity_abs = (
        cumulative_trapezoid(
            np.abs(current),
            time_s,
            initial=0.0,
        )
        / 3600
    )

    soc = (
        capacity_abs / Q_MEAS
        if charge
        else
        1.0 - capacity_abs / Q_MEAS
    )

    return {
        "t_min":
            time_s / 60,

        "V":
            voltage,

        "SOC":
            np.clip(
                soc,
                0,
                1,
            ),

        "Anode_potential_V":
            anode_potential,
    }


def interp_on_soc(
    soc,
    values,
    grid,
):

    soc = np.asarray(
        soc,
        dtype=float,
    )

    values = np.asarray(
        values,
        dtype=float,
    )

    valid = (
        np.isfinite(soc)
        & np.isfinite(values)
    )

    soc = soc[valid]
    values = values[valid]

    order = np.argsort(soc)
    soc = soc[order]
    values = values[order]

    unique_soc, index = np.unique(
        soc,
        return_index=True,
    )

    unique_values = values[index]

    output = np.full_like(
        grid,
        np.nan,
        dtype=float,
    )

    inside = (
        (grid >= unique_soc.min())
        & (grid <= unique_soc.max())
    )

    output[inside] = np.interp(
        grid[inside],
        unique_soc,
        unique_values,
    )

    return output


def voltage_on_fit_grid(
    rate,
    charge,
    pars,
):

    simulation = run_cc_dfn(
        rate,
        charge,
        pars,
    )

    model_voltage = interp_on_soc(
        simulation["SOC"],
        simulation["V"],
        FIT_SOC_GRID,
    )

    if not np.all(
        np.isfinite(model_voltage)
    ):
        raise RuntimeError(
            f"Model does not cover full FIT_SOC_GRID: "
            f"{rate}C {'Charge' if charge else 'Discharge'}"
        )

    return (
        model_voltage,
        simulation,
    )


STAGE1_BASELINE_SIMS = {}

for rate in ACTIVE_RATES:

    print(
        "Stage 1 baseline:",
        rate,
        "C Charge",
    )

    try:
        STAGE1_BASELINE_SIMS[
            (rate, True)
        ] = run_cc_dfn(
            rate,
            True,
            PARS_NOMINAL,
        )

    except Exception as error:
        print(
            "FAILED:",
            error,
        )


# Stage 2 — Charge-only sensitivity / identifiability

동적 parameter identification에는 **Charge 0.5C / 1C / 2C만 사용**한다.

- Stage 2A: `kn, kp, Dsn, Dsp, brugg_n, brugg_p, brugg_s`
- Stage 2B: `brugg_n = brugg_p = brugg_s = 1.5` 고정 후 `kn, kp, Dsn, Dsp` 재분석

Discharge dynamic data는 sensitivity, identifiability, subset selection에 사용하지 않는다.


In [ ]:
# ============================================================
# Charge sensitivity / identifiability helpers
# ============================================================

SENS_LOG_STEP = np.log(1.05)

BRUGG_POROSITY = {
    "brugg_n":
        float(
            AI2020_CORRECTED[
                "Negative electrode porosity"
            ]
        ),

    "brugg_p":
        float(
            AI2020_CORRECTED[
                "Positive electrode porosity"
            ]
        ),

    "brugg_s":
        float(
            AI2020_CORRECTED[
                "Separator porosity"
            ]
        ),
}


def perturb_for_lsa(
    base,
    parameter,
    direction,
):

    pars = base.copy()

    if parameter.startswith(
        "brugg_"
    ):

        porosity = (
            BRUGG_POROSITY[
                parameter
            ]
        )

        delta_b = (
            direction
            * SENS_LOG_STEP
            / np.log(porosity)
        )

        pars[parameter] = (
            float(base[parameter])
            + delta_b
        )

    else:

        pars[parameter] = (
            float(base[parameter])
            * np.exp(
                direction
                * SENS_LOG_STEP
            )
        )

    return pars


def experimental_voltage_on_fit_grid(
    rate,
    charge,
):

    experiment = EXP_DYN[
        (rate, charge)
    ]

    voltage = interp_on_soc(
        experiment["SOC"],
        experiment["V"],
        FIT_SOC_GRID,
    )

    if not np.all(
        np.isfinite(voltage)
    ):
        raise RuntimeError(
            f"Experiment does not cover full FIT_SOC_GRID: "
            f"{rate}C {'Charge' if charge else 'Discharge'}"
        )

    return voltage


def compute_charge_sensitivity(
    base,
    params,
    stage_name,
):

    columns = []
    detail_rows = []
    aggregate_rows = []

    for parameter in params:

        print(
            stage_name,
            parameter,
        )

        pars_plus = perturb_for_lsa(
            base,
            parameter,
            +1,
        )

        pars_minus = perturb_for_lsa(
            base,
            parameter,
            -1,
        )

        rate_blocks = []

        for rate in ACTIVE_RATES:

            voltage_plus, _ = voltage_on_fit_grid(
                rate,
                True,
                pars_plus,
            )

            voltage_minus, _ = voltage_on_fit_grid(
                rate,
                True,
                pars_minus,
            )

            sensitivity = (
                voltage_plus
                - voltage_minus
            ) / (
                2.0
                * SENS_LOG_STEP
            )

            rate_blocks.append(
                sensitivity
            )

            detail_rows.append({
                "Stage":
                    stage_name,

                "C-rate":
                    rate,

                "Parameter":
                    parameter,

                "Label":
                    PARAM_LABELS[
                        parameter
                    ],

                "RMS_mV_per_log_effect":
                    np.sqrt(
                        np.mean(
                            sensitivity**2
                        )
                    ) * 1000,

                "MaxAbs_mV_per_log_effect":
                    np.max(
                        np.abs(
                            sensitivity
                        )
                    ) * 1000,
            })

        combined = np.concatenate(
            rate_blocks
        )

        columns.append(
            combined
        )

        aggregate_rows.append({
            "Stage":
                stage_name,

            "Parameter":
                parameter,

            "Label":
                PARAM_LABELS[
                    parameter
                ],

            "RMS_mV_per_log_effect":
                np.sqrt(
                    np.mean(
                        combined**2
                    )
                ) * 1000,

            "MaxAbs_mV_per_log_effect":
                np.max(
                    np.abs(
                        combined
                    )
                ) * 1000,
        })

    matrix = np.column_stack(
        columns
    )

    return {
        "matrix":
            matrix,

        "detail":
            pd.DataFrame(
                detail_rows
            ),

        "aggregate":
            pd.DataFrame(
                aggregate_rows
            ),
    }


def identifiability_report(
    matrix,
    params,
):

    column_norm = np.linalg.norm(
        matrix,
        axis=0,
    )

    safe_norm = np.where(
        column_norm > 0,
        column_norm,
        1.0,
    )

    normalized = (
        matrix
        / safe_norm
    )

    cosine = (
        normalized.T
        @ normalized
    )

    singular_values = np.linalg.svd(
        normalized,
        compute_uv=False,
    )

    relative_sigma = (
        singular_values
        / singular_values[0]
    )

    condition_number = (
        np.inf
        if singular_values[-1] == 0
        else
        singular_values[0]
        / singular_values[-1]
    )

    singular_table = pd.DataFrame({
        "Index":
            np.arange(
                1,
                len(singular_values) + 1
            ),

        "Singular_value":
            singular_values,

        "Relative_sigma":
            relative_sigma,
    })

    return {
        "cosine":
            pd.DataFrame(
                cosine,
                index=params,
                columns=params,
            ),

        "singular_values":
            singular_table,

        "condition_number":
            condition_number,

        "rank_rel_1e2":
            int(
                np.sum(
                    relative_sigma
                    >= 1e-2
                )
            ),

        "rank_rel_1e3":
            int(
                np.sum(
                    relative_sigma
                    >= 1e-3
                )
            ),
    }


def greedy_subset(
    matrix,
    params,
    relative_sensitivity_floor=0.10,
    cosine_threshold=0.95,
):

    rms = np.sqrt(
        np.mean(
            matrix**2,
            axis=0,
        )
    )

    rms_rel = (
        rms
        / np.max(rms)
    )

    column_norm = np.linalg.norm(
        matrix,
        axis=0,
    )

    normalized = (
        matrix
        / np.where(
            column_norm > 0,
            column_norm,
            1.0,
        )
    )

    cosine = (
        normalized.T
        @ normalized
    )

    order = np.argsort(
        -rms
    )

    selected = []

    for index in order:

        if (
            rms_rel[index]
            < relative_sensitivity_floor
        ):
            continue

        if not selected:
            selected.append(
                index
            )
            continue

        max_collinearity = max(
            abs(
                cosine[
                    index,
                    selected_index
                ]
            )
            for selected_index
            in selected
        )

        if (
            max_collinearity
            < cosine_threshold
        ):
            selected.append(
                index
            )

    return [
        params[index]
        for index in selected
    ]


def save_identifiability(
    prefix,
    result,
):

    result["cosine"].to_csv(
        RESULTS_DIR
        / f"{prefix}_cosine.csv",
        encoding="utf-8-sig",
    )

    result[
        "singular_values"
    ].to_csv(
        RESULTS_DIR
        / f"{prefix}_singular.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pd.DataFrame([{
        "Condition_number":
            result[
                "condition_number"
            ],

        "Rank_rel_1e2":
            result[
                "rank_rel_1e2"
            ],

        "Rank_rel_1e3":
            result[
                "rank_rel_1e3"
            ],
    }]).to_csv(
        RESULTS_DIR
        / f"{prefix}_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )


## Stage 2A — 7-parameter Charge sensitivity / identifiability


In [ ]:
SENS_BASE_7 = PARS_NOMINAL.copy()

SENS7 = compute_charge_sensitivity(
    SENS_BASE_7,
    PARAMS_7,
    "Stage 2A",
)

S7 = SENS7["matrix"]
SENSITIVITY_7 = SENS7["aggregate"]

ID7 = identifiability_report(
    S7,
    PARAMS_7,
)

AUTO_SUBSET_7 = greedy_subset(
    S7,
    PARAMS_7,
)

display(
    SENSITIVITY_7
    .sort_values(
        "RMS_mV_per_log_effect",
        ascending=False,
    )
    .round(3)
)

print(
    "Condition number =",
    ID7["condition_number"],
)

print(
    "Rank (relative sigma >= 1e-2) =",
    ID7["rank_rel_1e2"],
)

print(
    "Rank (relative sigma >= 1e-3) =",
    ID7["rank_rel_1e3"],
)

display(
    ID7["cosine"].round(3)
)

display(
    ID7["singular_values"].round(5)
)

print(
    "Stage 2A auto subset =",
    AUTO_SUBSET_7,
)

SENS7["detail"].to_csv(
    RESULTS_DIR
    / "stage2a_sensitivity_7param_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

SENSITIVITY_7.to_csv(
    RESULTS_DIR
    / "stage2a_sensitivity_7param.csv",
    index=False,
    encoding="utf-8-sig",
)

save_identifiability(
    "stage2a_identifiability_7param",
    ID7,
)


## Stage 2B — Bruggeman = 1.5 고정 후 4-parameter Charge 재분석


In [ ]:
SENS_BASE_4 = PARS_BRUGG_15.copy()

SENS4 = compute_charge_sensitivity(
    SENS_BASE_4,
    PARAMS_4,
    "Stage 2B",
)

S4 = SENS4["matrix"]
SENSITIVITY_4 = SENS4["aggregate"]

ID4 = identifiability_report(
    S4,
    PARAMS_4,
)

AUTO_SUBSET_4 = greedy_subset(
    S4,
    PARAMS_4,
)

display(
    SENSITIVITY_4
    .sort_values(
        "RMS_mV_per_log_effect",
        ascending=False,
    )
    .round(3)
)

print(
    "Condition number =",
    ID4["condition_number"],
)

print(
    "Rank (relative sigma >= 1e-2) =",
    ID4["rank_rel_1e2"],
)

print(
    "Rank (relative sigma >= 1e-3) =",
    ID4["rank_rel_1e3"],
)

display(
    ID4["cosine"].round(3)
)

display(
    ID4["singular_values"].round(5)
)

print(
    "Stage 2B auto subset =",
    AUTO_SUBSET_4,
)

SENS4["detail"].to_csv(
    RESULTS_DIR
    / "stage2b_sensitivity_4param_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

SENSITIVITY_4.to_csv(
    RESULTS_DIR
    / "stage2b_sensitivity_4param.csv",
    index=False,
    encoding="utf-8-sig",
)

save_identifiability(
    "stage2b_identifiability_4param",
    ID4,
)


# Stage 3 — Charge-only fitting · 2개 알고리즘 비교

Primary parameterization case:

- `brugg_n = brugg_p = brugg_s = 1.5`
- Stage 2B에서 선택된 동일한 `AUTO_SUBSET_4`를 두 알고리즘 모두 사용
- 0.5C / 1C / 2C Charge만 objective에 사용
- 동일한 SOC 10–90% grid, bounds, OCP, stoichiometry를 사용

비교하는 최적화 전략:

### Algorithm A — Multi-start TRF
Bounded nonlinear least-squares에 직접 적합한 local optimizer.  
Nominal start + reproducible random start를 사용한다.

### Algorithm B — Differential Evolution → TRF
Differential Evolution으로 전역 영역을 탐색한 뒤, 동일한 TRF least-squares로 local refinement한다.

두 전략의 차이는 **초기값/전역 탐색 방식**이며, 최종 local objective와 physical model은 동일하다.

**중요:** 알고리즘별 parameter와 best run은 Charge fitting 성능만으로 결정한다.  
Discharge validation 결과를 보고 parameter를 다시 선택하거나 수정하지 않는다.


In [ ]:
# ============================================================
# Stage 3 configuration / parameter encoding
# ============================================================

USE_AUTO_SUBSET = True

OPT_BASE = PARS_BRUGG_15.copy()

OPT_PARAMS = (
    AUTO_SUBSET_4.copy()
    if (
        USE_AUTO_SUBSET
        and len(AUTO_SUBSET_4) > 0
    )
    else PARAMS_4.copy()
)

if len(OPT_PARAMS) == 0:
    raise RuntimeError(
        "No valid optimization parameter subset."
    )

print(
    "Stage 3 fitting parameters =",
    OPT_PARAMS,
)

print(
    "Bruggeman fixed =",
    OPT_BASE["brugg_n"],
    OPT_BASE["brugg_p"],
    OPT_BASE["brugg_s"],
)


def encode_params(
    pars,
    opt_params,
):

    return np.array([
        np.log(
            float(
                pars[parameter]
            )
        )
        for parameter
        in opt_params
    ])


def decode_params(
    z,
    base,
    opt_params,
):

    pars = base.copy()

    for parameter, value in zip(
        opt_params,
        z,
    ):

        pars[parameter] = float(
            np.exp(value)
        )

    return pars


def log_bounds(
    opt_params,
):

    lower = []
    upper = []

    for parameter in opt_params:

        lo, hi = PARAM_BOUNDS[
            parameter
        ]

        lower.append(
            np.log(lo)
        )

        upper.append(
            np.log(hi)
        )

    return (
        np.asarray(lower),
        np.asarray(upper),
    )


def bound_hit_string(
    pars,
    opt_params,
    tolerance=0.01,
):

    hits = []

    for parameter in opt_params:

        lo, hi = PARAM_BOUNDS[
            parameter
        ]

        value = float(
            pars[parameter]
        )

        span = hi - lo

        if (
            abs(value - lo)
            <= tolerance * span
        ):
            hits.append(
                parameter
                + ":lower"
            )

        elif (
            abs(value - hi)
            <= tolerance * span
        ):
            hits.append(
                parameter
                + ":upper"
            )

    return (
        ", ".join(hits)
        if hits
        else ""
    )


LOWER_LOG, UPPER_LOG = log_bounds(
    OPT_PARAMS
)

Z0 = encode_params(
    OPT_BASE,
    OPT_PARAMS,
)

FAIL_PENALTY_V = 0.25


In [ ]:
# ============================================================
# Stage 3 Charge fitting objective
# ============================================================

EXP_CHARGE_GRID = {
    rate:
        experimental_voltage_on_fit_grid(
            rate,
            True,
        )
    for rate in ACTIVE_RATES
}


def charge_condition_residual(
    rate,
    pars,
):

    model_voltage, _ = (
        voltage_on_fit_grid(
            rate,
            True,
            pars,
        )
    )

    return (
        model_voltage
        - EXP_CHARGE_GRID[rate]
    )


def global_charge_residual(
    z,
):

    pars = decode_params(
        z,
        OPT_BASE,
        OPT_PARAMS,
    )

    blocks = []

    for rate in ACTIVE_RATES:

        try:

            residual = (
                charge_condition_residual(
                    rate,
                    pars,
                )
            )

        except Exception as error:

            print(
                "Simulation failure:",
                rate,
                "C Charge |",
                error,
            )

            residual = np.full(
                len(FIT_SOC_GRID),
                FAIL_PENALTY_V,
            )

        blocks.append(
            residual
        )

    return np.concatenate(
        blocks
    )


def charge_fit_rmse_mV(
    pars,
):

    blocks = [
        charge_condition_residual(
            rate,
            pars,
        )
        for rate in ACTIVE_RATES
    ]

    error = np.concatenate(
        blocks
    )

    return (
        np.sqrt(
            np.mean(
                error**2
            )
        )
        * 1000
    )


def make_time_limit_callback(
    max_minutes,
):

    start_time = time.time()

    def callback(
        intermediate_result,
    ):

        elapsed_min = (
            time.time()
            - start_time
        ) / 60.0

        if (
            elapsed_min
            >= max_minutes
        ):

            print(
                "[STOP] per-start wall-time limit:",
                f"{elapsed_min:.1f} min",
            )

            raise StopIteration

    return callback


In [ ]:
# ============================================================
# Stage 3 — two-algorithm optimizer comparison
# ============================================================

def make_trf_time_limit_callback(
    max_minutes,
):
    start_time = time.time()

    def callback(
        intermediate_result,
    ):
        elapsed_min = (
            time.time()
            - start_time
        ) / 60.0

        if elapsed_min >= max_minutes:
            print(
                "[STOP] TRF wall-time limit:",
                f"{elapsed_min:.1f} min",
            )
            raise StopIteration

    return callback


def make_de_time_limit_callback(
    max_minutes,
):
    start_time = time.time()

    def callback(
        x,
        convergence,
    ):
        elapsed_min = (
            time.time()
            - start_time
        ) / 60.0

        if elapsed_min >= max_minutes:
            print(
                "[STOP] DE wall-time limit:",
                f"{elapsed_min:.1f} min",
            )
            return True

        return False

    return callback


def scalar_charge_objective(
    z,
):
    residual = global_charge_residual(
        z
    )

    return 0.5 * float(
        np.sum(
            residual**2
        )
    )


def build_candidate(
    algorithm,
    run,
    seed,
    z,
    nfev,
    success,
    status,
    message,
    elapsed_min,
):
    pars = decode_params(
        z,
        OPT_BASE,
        OPT_PARAMS,
    )

    fit_rmse = charge_fit_rmse_mV(
        pars
    )

    return {
        "Algorithm":
            algorithm,

        "Run":
            run,

        "Seed":
            seed,

        "pars":
            pars,

        "Fit_RMSE_mV":
            fit_rmse,

        "Objective":
            scalar_charge_objective(
                z
            ),

        "nfev":
            nfev,

        "Elapsed_min":
            elapsed_min,

        "Success":
            success,

        "Status":
            status,

        "Message":
            message,

        "Bound_hit":
            bound_hit_string(
                pars,
                OPT_PARAMS,
            ),
    }


def candidate_to_row(
    candidate,
):
    row = {
        key: value
        for key, value
        in candidate.items()
        if key != "pars"
    }

    for parameter in PARAMS_7:
        row[parameter] = float(
            candidate[
                "pars"
            ][parameter]
        )

    return row


ALL_CANDIDATES = []
CHECKPOINT_ROWS = []

stage3_start = time.time()


# ------------------------------------------------------------
# Algorithm A — Multi-start TRF
# ------------------------------------------------------------

if "TRF" in OPTIMIZATION_ALGORITHMS:

    starts = [
        np.clip(
            Z0,
            LOWER_LOG,
            UPPER_LOG,
        )
    ]

    for _ in range(
        max(
            0,
            TRF_STARTS - 1,
        )
    ):
        starts.append(
            RNG.uniform(
                LOWER_LOG,
                UPPER_LOG,
            )
        )

    for start_number, z_start in enumerate(
        starts,
        start=1,
    ):

        total_elapsed_min = (
            time.time()
            - stage3_start
        ) / 60.0

        if (
            total_elapsed_min
            >= MAX_TOTAL_STAGE3_WALL_MIN
        ):
            print(
                "[STOP] Stage 3 total wall-time limit reached."
            )
            break

        print()
        print(
            f"=== TRF start "
            f"{start_number}/{len(starts)} ==="
        )

        run_start = time.time()

        try:
            result = least_squares(
                global_charge_residual,
                x0=z_start,
                bounds=(
                    LOWER_LOG,
                    UPPER_LOG,
                ),
                method="trf",
                max_nfev=TRF_MAX_NFEV,
                xtol=1e-4,
                ftol=1e-4,
                gtol=1e-4,
                verbose=1,
                callback=
                    make_trf_time_limit_callback(
                        MAX_WALL_MIN_PER_TRF_START
                    ),
            )

            candidate = build_candidate(
                algorithm="TRF",
                run=start_number,
                seed=np.nan,
                z=result.x,
                nfev=int(
                    result.nfev
                ),
                success=bool(
                    result.success
                ),
                status=int(
                    result.status
                ),
                message=str(
                    result.message
                ),
                elapsed_min=(
                    time.time()
                    - run_start
                ) / 60.0,
            )

            ALL_CANDIDATES.append(
                candidate
            )

            print(
                "TRF Charge RMSE =",
                f"{candidate['Fit_RMSE_mV']:.3f}",
                "mV",
            )

        except Exception as error:
            CHECKPOINT_ROWS.append({
                "Algorithm":
                    "TRF",
                "Run":
                    start_number,
                "Seed":
                    np.nan,
                "Fit_RMSE_mV":
                    np.nan,
                "Objective":
                    np.nan,
                "nfev":
                    np.nan,
                "Elapsed_min":
                    (
                        time.time()
                        - run_start
                    ) / 60.0,
                "Success":
                    False,
                "Status":
                    np.nan,
                "Message":
                    repr(error),
                "Bound_hit":
                    "",
            })

            print(
                "[FAILED] TRF:",
                error,
            )

        if ALL_CANDIDATES:
            newest = ALL_CANDIDATES[-1]
            if (
                newest["Algorithm"]
                == "TRF"
                and newest["Run"]
                == start_number
            ):
                CHECKPOINT_ROWS.append(
                    candidate_to_row(
                        newest
                    )
                )

        pd.DataFrame(
            CHECKPOINT_ROWS
        ).to_csv(
            RESULTS_DIR
            / "optimization_starts_checkpoint.csv",
            index=False,
            encoding="utf-8-sig",
        )


# ------------------------------------------------------------
# Algorithm B — Differential Evolution -> TRF
# ------------------------------------------------------------

if "DE_TRF" in OPTIMIZATION_ALGORITHMS:

    bounds_list = list(
        zip(
            LOWER_LOG,
            UPPER_LOG,
        )
    )

    for seed in DE_SEEDS:

        total_elapsed_min = (
            time.time()
            - stage3_start
        ) / 60.0

        if (
            total_elapsed_min
            >= MAX_TOTAL_STAGE3_WALL_MIN
        ):
            print(
                "[STOP] Stage 3 total wall-time limit reached."
            )
            break

        print()
        print(
            f"=== DE -> TRF seed={seed} ==="
        )

        run_start = time.time()

        try:
            result_de = differential_evolution(
                scalar_charge_objective,
                bounds=bounds_list,
                seed=seed,
                maxiter=DE_MAXITER,
                popsize=DE_POPSIZE,
                polish=False,
                updating="immediate",
                workers=1,
                callback=
                    make_de_time_limit_callback(
                        MAX_WALL_MIN_PER_DE_SEED
                    ),
            )

            print(
                "DE finished | fun =",
                result_de.fun,
                "| nfev =",
                result_de.nfev,
            )

            result_refine = least_squares(
                global_charge_residual,
                x0=np.clip(
                    result_de.x,
                    LOWER_LOG,
                    UPPER_LOG,
                ),
                bounds=(
                    LOWER_LOG,
                    UPPER_LOG,
                ),
                method="trf",
                max_nfev=
                    DE_REFINE_MAX_NFEV,
                xtol=1e-4,
                ftol=1e-4,
                gtol=1e-4,
                verbose=1,
                callback=
                    make_trf_time_limit_callback(
                        MAX_WALL_MIN_PER_TRF_START
                    ),
            )

            candidate = build_candidate(
                algorithm="DE->TRF",
                run=np.nan,
                seed=seed,
                z=result_refine.x,
                nfev=int(
                    result_de.nfev
                    + result_refine.nfev
                ),
                success=bool(
                    result_refine.success
                ),
                status=int(
                    result_refine.status
                ),
                message=(
                    "DE: "
                    + str(
                        result_de.message
                    )
                    + " | TRF: "
                    + str(
                        result_refine.message
                    )
                ),
                elapsed_min=(
                    time.time()
                    - run_start
                ) / 60.0,
            )

            ALL_CANDIDATES.append(
                candidate
            )

            CHECKPOINT_ROWS.append(
                candidate_to_row(
                    candidate
                )
            )

            print(
                "DE->TRF Charge RMSE =",
                f"{candidate['Fit_RMSE_mV']:.3f}",
                "mV",
            )

        except Exception as error:
            CHECKPOINT_ROWS.append({
                "Algorithm":
                    "DE->TRF",
                "Run":
                    np.nan,
                "Seed":
                    seed,
                "Fit_RMSE_mV":
                    np.nan,
                "Objective":
                    np.nan,
                "nfev":
                    np.nan,
                "Elapsed_min":
                    (
                        time.time()
                        - run_start
                    ) / 60.0,
                "Success":
                    False,
                "Status":
                    np.nan,
                "Message":
                    repr(error),
                "Bound_hit":
                    "",
            })

            print(
                "[FAILED] DE->TRF:",
                error,
            )

        pd.DataFrame(
            CHECKPOINT_ROWS
        ).to_csv(
            RESULTS_DIR
            / "optimization_starts_checkpoint.csv",
            index=False,
            encoding="utf-8-sig",
        )


if not ALL_CANDIDATES:
    raise RuntimeError(
        "No valid optimization candidate. "
        "optimization_starts_checkpoint.csv를 확인하세요."
    )


# ------------------------------------------------------------
# Algorithm별 best는 Charge fitting RMSE로만 선택
# ------------------------------------------------------------

BEST_BY_ALGORITHM = {}

for algorithm in [
    "TRF",
    "DE->TRF",
]:

    candidates = [
        candidate
        for candidate
        in ALL_CANDIDATES
        if candidate[
            "Algorithm"
        ] == algorithm
    ]

    if candidates:
        BEST_BY_ALGORITHM[
            algorithm
        ] = min(
            candidates,
            key=lambda item:
                item[
                    "Fit_RMSE_mV"
                ],
        )


# 전체 primary best 역시 Charge fitting RMSE로만 선택
BEST = min(
    BEST_BY_ALGORITHM.values(),
    key=lambda item:
        item[
            "Fit_RMSE_mV"
        ],
)

BEST_PARS = BEST[
    "pars"
]

BEST_SCORE_MV = BEST[
    "Fit_RMSE_mV"
]

BEST_ALGORITHM = BEST[
    "Algorithm"
]


print()
print(
    "=== ALGORITHM BEST RESULTS ==="
)

for algorithm, candidate in (
    BEST_BY_ALGORITHM.items()
):
    print(
        algorithm,
        "| Charge RMSE =",
        f"{candidate['Fit_RMSE_mV']:.3f}",
        "mV",
        "| elapsed =",
        f"{candidate['Elapsed_min']:.1f}",
        "min",
    )


print()
print(
    "Primary best by Charge fitting only =",
    BEST_ALGORITHM,
)

print(
    "Charge fitting RMSE =",
    BEST_SCORE_MV,
    "mV",
)


# Algorithm별 parameter table
algorithm_parameter_rows = []

for algorithm, candidate in (
    BEST_BY_ALGORITHM.items()
):
    for parameter in PARAMS_7:
        algorithm_parameter_rows.append({
            "Algorithm":
                algorithm,
            "Parameter":
                parameter,
            "Value":
                candidate[
                    "pars"
                ][parameter],
            "Optimized":
                parameter
                in OPT_PARAMS,
            "Bound_hit":
                bound_hit_string(
                    candidate[
                        "pars"
                    ],
                    [parameter],
                )
                if parameter
                in OPT_PARAMS
                else "",
        })

ALGORITHM_PARAMETER_TABLE = (
    pd.DataFrame(
        algorithm_parameter_rows
    )
)

ALGORITHM_PARAMETER_TABLE.to_csv(
    RESULTS_DIR
    / "best_parameters_by_algorithm.csv",
    index=False,
    encoding="utf-8-sig",
)


BEST_PARAMETER_TABLE = (
    ALGORITHM_PARAMETER_TABLE[
        ALGORITHM_PARAMETER_TABLE[
            "Algorithm"
        ] == BEST_ALGORITHM
    ].copy()
)

BEST_PARAMETER_TABLE.to_csv(
    RESULTS_DIR
    / "best_parameters.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    ALGORITHM_PARAMETER_TABLE
)


# Stage 4 — Discharge held-out dynamic validation

Stage 3에서 최종 parameter를 확정한 뒤에만 Discharge dynamic profile을 평가한다.

- Charge: `Fitting`
- Discharge: `Validation`

동일한 parameter set으로 0.5C / 1C / 2C를 평가한다.
`SMOKE_TEST=True`일 때는 0.5C만 실행한다.


In [ ]:
# ============================================================
# Stage 4 validation metrics
# ============================================================

def interp_on_time(
    time_values,
    values,
    grid,
):

    time_values = np.asarray(
        time_values,
        dtype=float,
    )

    values = np.asarray(
        values,
        dtype=float,
    )

    valid = (
        np.isfinite(
            time_values
        )
        & np.isfinite(
            values
        )
    )

    time_values = (
        time_values[
            valid
        ]
    )

    values = values[
        valid
    ]

    order = np.argsort(
        time_values
    )

    time_values = (
        time_values[
            order
        ]
    )

    values = values[
        order
    ]

    unique_time, index = np.unique(
        time_values,
        return_index=True,
    )

    unique_values = (
        values[
            index
        ]
    )

    output = np.full_like(
        grid,
        np.nan,
        dtype=float,
    )

    inside = (
        (grid >= unique_time.min())
        & (grid <= unique_time.max())
    )

    output[inside] = np.interp(
        grid[inside],
        unique_time,
        unique_values,
    )

    return output


def evaluate_dynamic_condition(
    rate,
    charge,
    pars,
):

    experiment = EXP_DYN[
        (rate, charge)
    ]

    simulation = run_cc_dfn(
        rate,
        charge,
        pars,
    )

    voltage_exp_soc = (
        experimental_voltage_on_fit_grid(
            rate,
            charge,
        )
    )

    voltage_model_soc = interp_on_soc(
        simulation["SOC"],
        simulation["V"],
        FIT_SOC_GRID,
    )

    if not np.all(
        np.isfinite(
            voltage_model_soc
        )
    ):
        raise RuntimeError(
            "Simulation does not cover "
            "the full 10–90% SOC grid."
        )

    error_soc_mV = (
        voltage_model_soc
        - voltage_exp_soc
    ) * 1000


    t_start = max(
        np.nanmin(
            experiment["t_min"]
        ),
        np.nanmin(
            simulation["t_min"]
        ),
    )

    t_end = min(
        np.nanmax(
            experiment["t_min"]
        ),
        np.nanmax(
            simulation["t_min"]
        ),
    )

    time_grid = np.linspace(
        t_start,
        t_end,
        400,
    )

    voltage_exp_time = interp_on_time(
        experiment["t_min"],
        experiment["V"],
        time_grid,
    )

    voltage_model_time = interp_on_time(
        simulation["t_min"],
        simulation["V"],
        time_grid,
    )

    valid_time = (
        np.isfinite(
            voltage_exp_time
        )
        & np.isfinite(
            voltage_model_time
        )
    )

    error_time_mV = (
        voltage_model_time[
            valid_time
        ]
        - voltage_exp_time[
            valid_time
        ]
    ) * 1000


    return {
        "simulation":
            simulation,

        "RMSE_SOC10_90_mV":
            np.sqrt(
                np.mean(
                    error_soc_mV**2
                )
            ),

        "MAE_SOC10_90_mV":
            np.mean(
                np.abs(
                    error_soc_mV
                )
            ),

        "RMSE_full_overlap_mV":
            np.sqrt(
                np.mean(
                    error_time_mV**2
                )
            ),

        "MAE_full_overlap_mV":
            np.mean(
                np.abs(
                    error_time_mV
                )
            ),

        "End_time_error_min":
            (
                simulation[
                    "t_min"
                ][-1]
                - experiment[
                    "t_min"
                ][-1]
            ),

        "Coverage_fraction":
            float(
                np.mean(
                    np.isfinite(
                        voltage_model_soc
                    )
                )
            ),
    }


In [ ]:
# ============================================================
# Stage 4 — fitting / validation evaluation by algorithm
# ============================================================

FINAL_SIMS = {}
validation_rows = []
anode_rows = []


for algorithm, candidate in (
    BEST_BY_ALGORITHM.items()
):

    pars = candidate[
        "pars"
    ]

    for rate in ACTIVE_RATES:

        for charge in [
            True,
            False,
        ]:

            direction = (
                "Charge"
                if charge
                else "Discharge"
            )

            role = (
                "Fitting"
                if charge
                else "Validation"
            )

            print(
                algorithm,
                "|",
                rate,
                "C",
                direction,
                "|",
                role,
            )

            try:

                result = evaluate_dynamic_condition(
                    rate,
                    charge,
                    pars,
                )

                simulation = (
                    result.pop(
                        "simulation"
                    )
                )

                FINAL_SIMS[
                    (
                        algorithm,
                        rate,
                        charge,
                    )
                ] = simulation


                min_anode = np.nan
                soc_at_min = np.nan
                time_at_min = np.nan


                if charge:

                    min_index = int(
                        np.nanargmin(
                            simulation[
                                "Anode_potential_V"
                            ]
                        )
                    )

                    min_anode = float(
                        simulation[
                            "Anode_potential_V"
                        ][min_index]
                    )

                    soc_at_min = float(
                        simulation[
                            "SOC"
                        ][min_index]
                    )

                    time_at_min = float(
                        simulation[
                            "t_min"
                        ][min_index]
                    )


                    for (
                        t_min,
                        soc,
                        voltage,
                        anode_v,
                    ) in zip(
                        simulation["t_min"],
                        simulation["SOC"],
                        simulation["V"],
                        simulation[
                            "Anode_potential_V"
                        ],
                    ):

                        anode_rows.append({
                            "Algorithm":
                                algorithm,

                            "C-rate":
                                rate,

                            "Time_min":
                                t_min,

                            "SOC":
                                soc,

                            "Terminal_voltage_V":
                                voltage,

                            "Anode_potential_V":
                                anode_v,
                        })


                validation_rows.append({
                    "Algorithm":
                        algorithm,

                    "C-rate":
                        rate,

                    "Direction":
                        direction,

                    "Role":
                        role,

                    **result,

                    "Min_anode_V":
                        min_anode,

                    "SOC_at_min_anode":
                        soc_at_min,

                    "Time_at_min_anode_min":
                        time_at_min,

                    "Status":
                        "OK",
                })


            except Exception as error:

                print(
                    "FAILED:",
                    error,
                )

                validation_rows.append({
                    "Algorithm":
                        algorithm,

                    "C-rate":
                        rate,

                    "Direction":
                        direction,

                    "Role":
                        role,

                    "RMSE_SOC10_90_mV":
                        np.nan,

                    "MAE_SOC10_90_mV":
                        np.nan,

                    "RMSE_full_overlap_mV":
                        np.nan,

                    "MAE_full_overlap_mV":
                        np.nan,

                    "End_time_error_min":
                        np.nan,

                    "Coverage_fraction":
                        np.nan,

                    "Min_anode_V":
                        np.nan,

                    "SOC_at_min_anode":
                        np.nan,

                    "Time_at_min_anode_min":
                        np.nan,

                    "Status":
                        "FAIL",
                })


VALIDATION_TABLE = pd.DataFrame(
    validation_rows
)

display(
    VALIDATION_TABLE.round(4)
)


VALIDATION_TABLE.to_csv(
    RESULTS_DIR
    / "fit_validation_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)


ANODE_TABLE = pd.DataFrame(
    anode_rows
)

ANODE_TABLE.to_csv(
    RESULTS_DIR
    / "anode_potential_charge.csv",
    index=False,
    encoding="utf-8-sig",
)


algorithm_summary_rows = []

for algorithm in (
    BEST_BY_ALGORITHM.keys()
):

    table_algorithm = (
        VALIDATION_TABLE[
            VALIDATION_TABLE[
                "Algorithm"
            ] == algorithm
        ]
    )

    fit_rows = (
        table_algorithm[
            table_algorithm[
                "Role"
            ] == "Fitting"
        ]
    )

    val_rows = (
        table_algorithm[
            table_algorithm[
                "Role"
            ] == "Validation"
        ]
    )

    charge_rmse = np.sqrt(
        np.nanmean(
            fit_rows[
                "RMSE_SOC10_90_mV"
            ].to_numpy(
                dtype=float
            ) ** 2
        )
    )

    discharge_rmse = np.sqrt(
        np.nanmean(
            val_rows[
                "RMSE_SOC10_90_mV"
            ].to_numpy(
                dtype=float
            ) ** 2
        )
    )

    candidate = (
        BEST_BY_ALGORITHM[
            algorithm
        ]
    )

    algorithm_summary_rows.append({
        "Algorithm":
            algorithm,

        "Charge_fit_RMSE_mV":
            charge_rmse,

        "Discharge_validation_RMSE_mV":
            discharge_rmse,

        "Optimization_elapsed_min":
            candidate[
                "Elapsed_min"
            ],

        "Optimization_nfev":
            candidate[
                "nfev"
            ],

        "Bound_hit":
            candidate[
                "Bound_hit"
            ],

        "Selected_as_primary":
            algorithm
            == BEST_ALGORITHM,
    })


ALGORITHM_COMPARISON = pd.DataFrame(
    algorithm_summary_rows
)

ALGORITHM_COMPARISON.to_csv(
    RESULTS_DIR
    / "algorithm_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    ALGORITHM_COMPARISON.round(4)
)

print()
print(
    "Primary best was selected using Charge fitting only:",
    BEST_ALGORITHM,
)


In [ ]:
# ============================================================
# Stage 4 plots — algorithm comparison
# ============================================================

for algorithm in (
    BEST_BY_ALGORITHM.keys()
):

    fig, axes = plt.subplots(
        2,
        len(ACTIVE_RATES),
        figsize=(
            4.8
            * len(ACTIVE_RATES),
            7.5,
        ),
        squeeze=False,
        sharey=True,
    )

    for col, rate in enumerate(
        ACTIVE_RATES
    ):

        for row, charge in enumerate(
            [True, False]
        ):

            ax = axes[
                row,
                col
            ]

            experiment = EXP_DYN[
                (rate, charge)
            ]

            ax.plot(
                experiment["SOC"],
                experiment["V"],
                lw=2.4,
                label="Experiment",
            )

            key = (
                algorithm,
                rate,
                charge,
            )

            if key in FINAL_SIMS:

                simulation = (
                    FINAL_SIMS[
                        key
                    ]
                )

                ax.plot(
                    simulation["SOC"],
                    simulation["V"],
                    lw=1.8,
                    label=algorithm,
                )

            role = (
                "Fitting"
                if charge
                else "Validation"
            )

            ax.set_title(
                f"{algorithm} | "
                f"{rate}C "
                f"{'Charge' if charge else 'Discharge'}\n"
                f"{role}"
            )

            ax.set_xlabel(
                "SOC"
            )

            ax.set_ylabel(
                "Terminal voltage [V]"
            )

            ax.legend(
                fontsize=8
            )

    plt.tight_layout()

    safe_name = (
        algorithm
        .replace(
            ">",
            ""
        )
        .replace(
            "-",
            "_"
        )
    )

    figure_path = (
        RESULTS_DIR
        / (
            "optimized_fit_validation_"
            + safe_name
            + ".png"
        )
    )

    fig.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight",
    )

    print(
        "Saved:",
        figure_path,
    )

    plt.show()


# ------------------------------------------------------------
# Anode potential comparison
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(ACTIVE_RATES),
    figsize=(
        4.8
        * len(ACTIVE_RATES),
        4.2,
    ),
    squeeze=False,
)

for col, rate in enumerate(
    ACTIVE_RATES
):

    ax = axes[
        0,
        col
    ]

    for algorithm in (
        BEST_BY_ALGORITHM.keys()
    ):

        key = (
            algorithm,
            rate,
            True,
        )

        if key not in FINAL_SIMS:
            continue

        simulation = (
            FINAL_SIMS[
                key
            ]
        )

        ax.plot(
            simulation["SOC"],
            simulation[
                "Anode_potential_V"
            ],
            lw=1.8,
            label=algorithm,
        )

    ax.axhline(
        0,
        ls="--",
        lw=1,
    )

    ax.set_title(
        f"{rate}C Charge"
    )

    ax.set_xlabel(
        "SOC"
    )

    ax.set_ylabel(
        "Anode potential [V vs. Li/Li+]"
    )

    ax.legend(
        fontsize=8
    )

plt.tight_layout()

FIG_ANODE = (
    RESULTS_DIR
    / "anode_potential_charge_algorithm_comparison.png"
)

fig.savefig(
    FIG_ANODE,
    dpi=200,
    bbox_inches="tight",
)

print(
    "Saved:",
    FIG_ANODE,
)

plt.show()


In [ ]:
# ============================================================
# Final run summary
# ============================================================

RUN_SUMMARY = pd.DataFrame([{
    "Timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "Smoke_test":
        SMOKE_TEST,

    "Active_rates":
        ", ".join(
            str(rate)
            for rate
            in ACTIVE_RATES
        ),

    "Algorithms":
        ", ".join(
            BEST_BY_ALGORITHM.keys()
        ),

    "Primary_best_algorithm":
        BEST_ALGORITHM,

    "Primary_best_selection_rule":
        "Charge fitting RMSE only",

    "OCP_negative":
        "UN_EXP",

    "OCP_positive":
        "UP_EXP",

    "Stoichiometry_method":
        "1A endpoint-constrained",

    "Dynamic_identification_data":
        "Charge only",

    "Dynamic_validation_data":
        "Discharge",

    "Bruggeman_n":
        BEST_PARS[
            "brugg_n"
        ],

    "Bruggeman_p":
        BEST_PARS[
            "brugg_p"
        ],

    "Bruggeman_s":
        BEST_PARS[
            "brugg_s"
        ],

    "Selected_subset":
        ", ".join(
            OPT_PARAMS
        ),

    "Stage2A_condition_number":
        ID7[
            "condition_number"
        ],

    "Stage2B_condition_number":
        ID4[
            "condition_number"
        ],

    "TRF_starts":
        TRF_STARTS,

    "TRF_max_nfev":
        TRF_MAX_NFEV,

    "DE_seeds":
        ", ".join(
            str(seed)
            for seed in DE_SEEDS
        ),

    "DE_maxiter":
        DE_MAXITER,

    "DE_popsize":
        DE_POPSIZE,

    "DE_refine_max_nfev":
        DE_REFINE_MAX_NFEV,

    "Max_wall_min_per_TRF_start":
        MAX_WALL_MIN_PER_TRF_START,

    "Max_wall_min_per_DE_seed":
        MAX_WALL_MIN_PER_DE_SEED,

    "Max_total_stage3_wall_min":
        MAX_TOTAL_STAGE3_WALL_MIN,
}])


RUN_SUMMARY.to_csv(
    RESULTS_DIR
    / "run_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


display(
    RUN_SUMMARY.T
)

display(
    ALGORITHM_COMPARISON.T
)

print()
print(
    "All outputs saved to:",
    RESULTS_DIR,
)

if SMOKE_TEST:
    print(
        "\nSMOKE_TEST=True입니다. "
        "이 결과는 과학적 fitting 결과로 사용하지 마세요. "
        "실제 실행 전 SMOKE_TEST=False로 변경하세요."
    )


## 결과 확인 순서

실제 full run(`SMOKE_TEST=False`)이 끝난 뒤 아래 순서로 확인한다.

1. `stage1_stoichiometry.csv`
   - endpoint error
   - qOCV RMSE
   - `x0, x100, y0, y100`

2. `stage2a_sensitivity_7param.csv`
   - 7개 parameter sensitivity 규모
   - Bruggeman 포함 시 식별성

3. `stage2b_sensitivity_4param.csv`
   - Bruggeman = 1.5 이후 `kn, kp, Dsn, Dsp`
   - `AUTO_SUBSET_4`가 물리적으로 납득 가능한지

4. `optimization_starts_checkpoint.csv`
   - TRF start 및 DE→TRF 결과
   - Charge RMSE, nfev, 시간, bound hit, 종료 상태

5. `best_parameters_by_algorithm.csv`
   - TRF와 DE→TRF가 유사한 parameter 영역으로 수렴하는지

6. `algorithm_comparison.csv`
   - **Charge fitting RMSE**
   - **Discharge validation RMSE**
   - 계산시간 / nfev / bound hit
   - primary best는 Charge fitting 성능만으로 선택되었는지 확인

7. `fit_validation_metrics.csv`
   - C-rate별 Charge fitting / Discharge validation 오차

8. `optimized_fit_validation_*.png`
   - 각 알고리즘 parameter가 Charge와 Discharge 곡선 형상을 설명하는지

9. `anode_potential_charge_algorithm_comparison.png`
   - 두 알고리즘의 Charge anode-potential prediction 차이
   - minimum anode potential의 안정성

### 알고리즘 비교 해석

첫 비교에서는 **Multi-start TRF**와 **DE→TRF** 두 전략만 비교한다.

- TRF가 더 빠르고 두 전략이 비슷한 parameter / RMSE로 수렴하면 TRF를 기본 방법으로 채택할 근거가 생긴다.
- DE→TRF가 반복적으로 더 낮은 Charge objective를 찾고 parameter도 다른 영역에 위치한다면 local minimum / 초기값 의존성을 추가 검토한다.
- Discharge validation이 더 좋은 알고리즘으로 parameter를 사후 재선택하지 않는다. Discharge는 held-out validation 지표로만 해석한다.
